In [0]:
%run "/Workspace/Users/dungdq.b22kh019@stu.ptit.edu.vn/community_detection"

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


Tổng số cặp similarity: 3225710


Threshold (97th percentile): 0.7343
Số cạnh sau lọc: 99014


In [0]:
%pip install networkx
import networkx as nx
import numpy as np
import pandas as pd
from collections import defaultdict
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


Graph: 3256 nodes, 99014 edges


Số communities phát hiện được: 111


Modularity score: 0.5326



Top 10 communities lớn nhất:
community_id
35    260
16    210
41    198
28    195
19    187
14    169
0     153
24    153
23    147
27    143
Name: community_size, dtype: int64


Đã lưu community results


In [0]:
def random_walk_with_restart(G, start_nodes, alpha=0.15, max_iter=100, tol=1e-6):
    nodes     = list(G.nodes())
    n         = len(nodes)
    node_idx  = {node: i for i, node in enumerate(nodes)}
 
    from scipy.sparse import lil_matrix, csr_matrix
 
    W = lil_matrix((n, n), dtype=np.float64)
    for u, v, data in G.edges(data=True):
        w = data.get('weight', 1.0)
        if u in node_idx and v in node_idx:
            W[node_idx[u], node_idx[v]] = w
            W[node_idx[v], node_idx[u]] = w
    W = csr_matrix(W)
 
    row_sums = np.array(W.sum(axis=1)).flatten()
    row_sums[row_sums == 0] = 1.0
    D_inv = 1.0 / row_sums

    r = np.zeros(n, dtype=np.float64)
    valid_starts  = [node_idx[s] for s in start_nodes if s in node_idx]
    if not valid_starts:
        return {}
    r[valid_starts] = 1.0 / len(valid_starts)
 
    p = r.copy()
 
    for iteration in range(max_iter):
        p_new = (1 - alpha) * (W.T.dot(p * D_inv)) + alpha * r
 
        p_sum = p_new.sum()
        if p_sum > 1e-10:
            p_new = p_new / p_sum
        else:
            p_new = r.copy()   
 
        if np.linalg.norm(p_new - p, 1) < tol:
            break
        p = p_new
 
    p = np.clip(p, 0.0, 1.0)
 
    return {nodes[i]: float(p[i]) for i in range(n)}
 
 
def safe_normalize(x):
    x_min = x.min()
    x_max = x.max()
    spread = x_max - x_min
    if spread < 1e-10:
        return pd.Series(np.zeros(len(x)), index=x.index)
    return (x - x_min) / spread

In [0]:

from datetime import datetime
from collections import defaultdict
 
print("Tính Gateway scores...")
start_time = datetime.now()
 
community_to_nodes = defaultdict(list)
for sub, comm_id in partition.items():
    community_to_nodes[comm_id].append(sub)
 
large_communities = {
    c: nodes
    for c, nodes in community_to_nodes.items()
    if len(nodes) >= 3
}
print(f"Số communities >= 3 nodes: {len(large_communities)}")
 
gateway_results = []
all_nodes       = list(G.nodes())
 
for comm_id, comm_nodes in large_communities.items():
    outside_nodes = [n for n in all_nodes if partition.get(n) != comm_id]
 
    if len(outside_nodes) < 10:
        continue
 
    if len(outside_nodes) > 500:
        np.random.seed(42)
        outside_nodes = list(np.random.choice(outside_nodes, 500, replace=False))
 
    visit_probs = random_walk_with_restart(G, outside_nodes, alpha=0.15)
 
    for node in comm_nodes:
        if node in visit_probs:
            score = visit_probs[node]
            # Sanity check: score phải trong [0,1]
            assert 0.0 <= score <= 1.0 + 1e-9, \
                f"Score ngoài [0,1]: {node} = {score}"
            gateway_results.append({
                "subreddit"  : node,
                "community_id"  : comm_id,
                "community_size": len(comm_nodes),
                "gateway_score" : score,
                "role"          : "gateway"
            })
 
df_gateway = pd.DataFrame(gateway_results)
 
df_gateway["gateway_score_normalized"] = (
    df_gateway
    .groupby("community_id")["gateway_score"]
    .transform(safe_normalize)
)

df_gateway["gateway_score_normalized_global"] = safe_normalize(
    df_gateway["gateway_score"]
)
 
df_top_gateways = (
    df_gateway
    .sort_values("gateway_score", ascending=False)
    .groupby("community_id")
    .head(3)
)
 
elapsed = (datetime.now() - start_time).seconds
print(f"\nGateway tính xong: {elapsed}s")
print(f"Tổng gateway candidates: {len(df_gateway)}")
print(f"\nTop 10 gateway nodes:")
print(
    df_gateway
    .sort_values("gateway_score", ascending=False)
    .head(10)[[
        "subreddit",
        "community_id",
        "gateway_score",
        "gateway_score_normalized",
        "gateway_score_normalized_global"
    ]]
    .to_string(index=False)
)
 
# Kiểm tra nhanh giá trị
print(f"\nGateway score stats:")
print(f"  Min : {df_gateway['gateway_score'].min():.6f}")
print(f"  Max : {df_gateway['gateway_score'].max():.6f}")
print(f"  Mean: {df_gateway['gateway_score'].mean():.6f}")
 

Tính Gateway scores...
Số communities >= 3 nodes: 63

Gateway tính xong: 30s
Tổng gateway candidates: 3166

Top 10 gateway nodes:
          subreddit  community_id  gateway_score  gateway_score_normalized  gateway_score_normalized_global
        AgencySquad            16       0.001335                  1.000000                         1.000000
 ATBandATGcommunity            88       0.001227                  1.000000                         0.918851
          AnimeMeme            16       0.001223                  0.916368                         0.916380
          Bombstrap             0       0.001004                  1.000000                         0.752343
   CinnamonToastKen             0       0.001003                  0.999172                         0.751721
BikiniBottomTwitter             0       0.000957                  0.952888                         0.716941
         AlanBecker            56       0.000956                  1.000000                         0.716351
      

In [0]:
print("\nTính Bridge scores:")
start_time_br = datetime.now()
 
bridge_results  = []
community_ids   = list(large_communities.keys())
community_ids_sorted = sorted(
    community_ids,
    key=lambda c: len(large_communities[c]),
    reverse=True
)
 
for i, comm_x in enumerate(community_ids_sorted[:50]):
    nodes_x     = large_communities[comm_x]
    visit_probs = random_walk_with_restart(G, nodes_x, alpha=0.15)
 
    for node, score in visit_probs.items():
        node_community = partition.get(node)
        if node_community != comm_x and score > 0.001:
            bridge_results.append({
                "subreddit"          : node,
                "source_community"   : comm_x,
                "target_community"   : node_community,
                "bridge_score"       : score,
                "role"               : "bridge"
            })
 
    if (i + 1) % 10 == 0:
        elapsed = (datetime.now() - start_time_br).seconds
        print(f"  [{i+1}/50] {elapsed}s elapsed")
 
df_bridge = pd.DataFrame(bridge_results) if bridge_results else pd.DataFrame()
 
if not df_bridge.empty:
    df_bridge = df_bridge.sort_values("bridge_score", ascending=False)
 
    assert df_bridge["bridge_score"].between(0, 1 + 1e-9).all(), \
        "BUG: bridge_score ngoài [0,1]!"
 
    print(f"Tổng bridge candidates: {len(df_bridge)}")
    print("\nTop 10 bridges:")
    print(
        df_bridge
        .head(10)[["subreddit", "source_community", "target_community", "bridge_score"]]
        .to_string(index=False)
    )
    print(f"\nBridge score stats:")
    print(f"  Min : {df_bridge['bridge_score'].min():.6f}")
    print(f"  Max : {df_bridge['bridge_score'].max():.6f}")


Tính Bridge scores:
  [10/50] 4s elapsed
  [20/50] 9s elapsed
  [30/50] 13s elapsed
  [40/50] 19s elapsed
  [50/50] 24s elapsed
Tổng bridge candidates: 2576

Top 10 bridges:
           subreddit  source_community  target_community  bridge_score
             ChatGPT                72                 0      0.064398
  BallbustingStories                32                 7      0.037317
       CatholicMemes                17                 0      0.029758
CharacterActionGames                84                35      0.028462
       AliensAndUFOs                48                24      0.025207
       CelebRoleplay                33                 9      0.024529
        CanberraNSFW                33                 9      0.022920
    CharacterAi_NSFW                72                 0      0.022015
        ButchSelfies                91                14      0.021181
             ChaiApp                72                 0      0.020693

Bridge score stats:
  Min : 0.001000
  Max 

In [0]:
print("Phân tích overlap bridge và gateway")

if not df_bridge.empty:
    top_bridges = set(df_bridge.nlargest(100, "bridge_score")["subreddit"].tolist())

    top_gateways = set(df_top_gateways["subreddit"].tolist())
    
    overlap = top_bridges.intersection(top_gateways)
    overlap_pct = len(overlap) / len(top_bridges) * 100 if top_bridges else 0
    
    print(f"Top 100 bridges: {len(top_bridges)}")
    print(f"Top gateways: {len(top_gateways)}")
    print(f"Overlap: {len(overlap)} nodes ({overlap_pct:.1f}%)")
    print(f"Số lượng {overlap_pct:.1f}% bridges cũng là gateways")

Phân tích overlap bridge và gateway
Top 100 bridges: 100
Top gateways: 189
Overlap: 6 nodes (6.0%)
Số lượng 6.0% bridges cũng là gateways


In [0]:
df_gateway_spark = spark.createDataFrame(df_gateway)
df_gateway_spark.coalesce(1).write.mode("overwrite").option("header", True).csv(
    f"wasbs://{container}@{storage_account}.blob.core.windows.net/gateway_results"
)

if not df_bridge.empty:
    df_bridge_spark = spark.createDataFrame(df_bridge)
    df_bridge_spark.coalesce(1).write.mode("overwrite").option("header", True).csv(
        f"wasbs://{container}@{storage_account}.blob.core.windows.net/bridge_results"
    )

print("Đã lưu kết quả gateway_bridge")

Đã lưu kết quả gateway_bridge
